# Day 2 - MongoDB Atlas
## Semantic Recipe Finder

MongoDB Atlas is a fully managed cloud database built on MongoDB, the world's most popular document database. Atlas Vector Search adds native vector similarity search to Atlas collections, letting you embed and query unstructured content alongside your existing document data.

**When would you reach for this?**
- Your application already stores data as documents in MongoDB
- You want vector search without leaving the document model
- You need to search over rich, variably structured content

**The use case:** A recipe finder where users describe what they feel like eating in natural language - "something warm and spicy for a cold evening" or "a light summer dish with fresh vegetables" - and get semantically relevant recipes in return.

## 1. Setup

### Prerequisites

- A MongoDB Atlas account
- Ollama running locally with the `all-minilm` model pulled
- Python 3.12 with a virtual environment

### Create a Free MongoDB Atlas Cluster

If you do not already have a cluster:

1. Go to [MongoDB Atlas](https://www.mongodb.com/cloud/atlas/register) and create a free account
2. Once logged-in, select **Create a free starter database**
3. Give the cluster a name (e.g. `recipes`)
4. Click **Set it up for me**
5. Copy and save database user credentials
6. Select **Choose a connection method > Drivers > Python**
7. Note down the connection string that looks like `mongodb+srv://<username>:<password>@recipes.xxxxx.mongodb.net`
8. From the left navigation pane select **SECURITY > Database & Network Access**
9. From the left navigation pane select **NETWORK ACCESS > IP Access List**
10. Add `0.0.0.0/0` for temporary open access during development

### Set Environment Variable

Set your MongoDB connection string as an environment variable before running the notebook:

```shell
export MONGODB_URI="mongodb+srv://<username>:<password>@recipes.xxxxx.mongodb.net"
```

### Install Python Dependencies

In [1]:
%pip install ollama==0.6.2 \
             pandas==3.0.3 \
             pymongo==4.13.0 \
             tqdm==4.67.1 --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import ollama
import os
import random
import pandas as pd
import time

from ollama import ResponseError
from pymongo import MongoClient
from pymongo.operations import SearchIndexModel
from tqdm.notebook import tqdm

### Configuration

Replace `MONGODB_URI` with your connection string from the Atlas UI. `NUM_RECIPES` controls the size of the generated dataset - 200 is a reasonable default.

In [3]:
MONGODB_URI   = os.environ["MONGODB_URI"]
DB_NAME       = "recipes_db"
COLLECTION    = "recipes"
INDEX_NAME    = "vector_index"
LLM_EMBEDDING = "all-minilm"
NUM_RECIPES   = 200
RANDOM_SEED   = 42

> **Note:** `NUM_RECIPES` controls the size of the generated dataset. 200 is the recommended default for this tutorial - embedding generation runs locally via Ollama and is single-threaded, so larger values will work but will take proportionally longer. At 1,000 recipes expect a few minutes; at 20,000 expect significantly longer. Production pipelines would typically use a hosted embedding endpoint with async or batched generation to handle scale.

### Verify Ollama is running

In [4]:
ollama_ready = False

try:
    models = ollama.list()
    model_names = [m.model for m in models.models]
    assert any(LLM_EMBEDDING in m for m in model_names)
    print(f"Model '{LLM_EMBEDDING}' is ready.")
    ollama_ready = True
except ConnectionError:
    print("ERROR: Ollama is not running. Start it with: ollama serve")
except AssertionError:
    print(f"ERROR: Model not found. Run: ollama pull {LLM_EMBEDDING}")

Model 'all-minilm' is ready.


In [5]:
assert ollama_ready, "Please fix the Ollama issue above before continuing."

### Connect to MongoDB Atlas

In [6]:
client     = MongoClient(MONGODB_URI)
db         = client[DB_NAME]
collection = db[COLLECTION]

# Verify connection
client.admin.command("ping")
print("Connected to MongoDB Atlas.")

Connected to MongoDB Atlas.


## 2. The Dataset

We generate recipes programmatically from pools of cuisines, cooking methods, ingredients and description templates. Each recipe is a document - a natural fit for MongoDB's document model, since recipes have variably structured ingredient lists and step counts that do not map cleanly to fixed relational columns.

The `description` field is a short prose summary of the dish - this is what we embed and search over. The other fields serve as structured filters.

In [7]:
random.seed(RANDOM_SEED)

CUISINES = [
    "Italian", "Mexican", "Japanese", "Indian", "Thai",
    "French", "Chinese", "Mediterranean", "American", "Middle Eastern",
    "Korean", "Vietnamese", "Greek", "Spanish", "Ethiopian",
]

DIFFICULTIES = ["Easy", "Medium", "Hard"]

PREP_TIMES = [15, 20, 30, 45, 60, 90, 120]

PROTEINS = [
    "chicken", "beef", "lamb", "salmon", "tuna", "shrimp",
    "tofu", "chickpeas", "lentils", "eggs", "halloumi",
]

VEGETABLES = [
    "spinach", "roasted peppers", "zucchini", "eggplant", "tomatoes",
    "mushrooms", "broccoli", "sweet potato", "cauliflower", "kale",
    "asparagus", "green beans", "corn", "peas", "carrots",
]

BASES = [
    "rice", "pasta", "noodles", "flatbread", "couscous",
    "quinoa", "polenta", "crusty bread", "tortillas", "lentils",
]

COOKING_METHODS = [
    "slow-cooked", "grilled", "roasted", "pan-fried", "steamed",
    "braised", "baked", "stir-fried", "poached", "simmered",
]

FLAVORS = [
    "smoky and rich", "light and fresh", "warm and spicy", "tangy and bright",
    "creamy and comforting", "sweet and savory", "bold and aromatic",
    "delicate and fragrant", "hearty and satisfying", "zesty and vibrant",
]

OCCASIONS = [
    "a weeknight dinner", "a weekend feast", "meal prep", "a dinner party",
    "a light lunch", "a cold winter evening", "a warm summer day",
    "a quick family meal", "entertaining guests", "a cozy night in",
]

DESCRIPTION_TEMPLATES = [
    "{method} {protein} served over {base} with {vegetable}, {flavor} and perfect for {occasion}.",
    "A {flavor} {cuisine} dish featuring {method} {protein} with {vegetable} on a bed of {base}.",
    "{cuisine} classic: {method} {protein} with {vegetable}, {flavor} flavors ideal for {occasion}.",
    "Simple and satisfying - {method} {protein} tossed with {vegetable} and {base}, {flavor}.",
    "A {occasion} favorite: {flavor} {method} {protein} with {vegetable} and {base}.",
    "{method} {protein} paired with {vegetable} over {base} - a {flavor} {cuisine} inspired meal.",
    "Bring {cuisine} flavors home with this {method} {protein} and {vegetable} dish served with {base}.",
    "Quick and {flavor}: {method} {protein} with {vegetable} over {base}, great for {occasion}.",
]

RECIPE_NAME_TEMPLATES = [
    "{method} {protein} with {vegetable}",
    "{cuisine} {protein} and {vegetable}",
    "{method} {protein} over {base}",
    "{cuisine} style {protein} with {base}",
    "{protein} and {vegetable} {base} bowl",
]

def generate_recipe() -> dict:
    cuisine  = random.choice(CUISINES)
    method   = random.choice(COOKING_METHODS)
    protein  = random.choice(PROTEINS)
    veg      = random.choice(VEGETABLES)
    base     = random.choice(BASES)
    flavor   = random.choice(FLAVORS)
    occasion = random.choice(OCCASIONS)

    name = random.choice(RECIPE_NAME_TEMPLATES).format(
        cuisine = cuisine, method = method.capitalize(),
        protein = protein, vegetable = veg, base = base
    )

    description = random.choice(DESCRIPTION_TEMPLATES).format(
        cuisine = cuisine, method = method, protein = protein,
        vegetable = veg, base = base, flavor = flavor, occasion = occasion
    )

    prep_time  = random.choice(PREP_TIMES)
    difficulty = random.choice(DIFFICULTIES)

    num_ingredients = random.randint(5, 12)
    ingredient_pool = [protein, veg, base] + random.sample(VEGETABLES + PROTEINS, num_ingredients - 3)

    return {
        "name":        name.title(),
        "cuisine":     cuisine,
        "difficulty":  difficulty,
        "prep_time":   prep_time,
        "ingredients": list(set(ingredient_pool)),
        "description": description,
    }

recipes = [generate_recipe() for _ in range(NUM_RECIPES)]
df = pd.DataFrame(recipes)
print(f"Generated {len(df)} recipes")
df.head()

Generated 200 recipes


,name,cuisine,difficulty,prep_time,ingredients,description
0,Korean Chicken And Green Beans,Korean,Hard,90,"[peas, chicken, green beans, roasted peppers, ...",A tangy and bright Korean dish featuring grill...
1,Salmon And Eggplant Tortillas Bowl,Italian,Hard,90,"[peas, asparagus, mushrooms, tortillas, carrot...",Simple and satisfying - grilled salmon tossed ...
2,Thai Style Salmon With Quinoa,Thai,Medium,30,"[lentils, lamb, roasted peppers, corn, carrots...",A light and fresh Thai dish featuring roasted ...
3,Beef And Cauliflower Couscous Bowl,Ethiopian,Easy,90,"[tofu, couscous, beef, sweet potato, cauliflower]",Simple and satisfying - baked beef tossed with...
4,Steamed Beef Over Flatbread,Greek,Medium,90,"[peas, flatbread, green beans, tofu, beef, bro...",Quick and light and fresh: steamed beef with p...


## 3. Generate Embeddings and Load Data

We embed each recipe description and store the resulting vector as an `embedding` field on the document. This is the field our vector index will search over.

Note that MongoDB documents are schema-flexible - the `ingredients` field is a list of varying length, which slots naturally into the document model without any schema changes.

In [8]:
def get_embedding(text: str) -> list:
    response = ollama.embeddings(model = LLM_EMBEDDING, prompt = text)
    return response["embedding"]

# Verify embedding dimensions
test_embedding = get_embedding("a warm spicy dish with chicken")
EMBEDDING_DIMS = len(test_embedding)
print(f"Embedding dimensions: {EMBEDDING_DIMS}")

Embedding dimensions: 384


In [9]:
# Prepare for a clean run
collection.drop_search_index(INDEX_NAME)
collection.delete_many({})

documents = []
for recipe in tqdm(recipes, desc = "Generating embeddings"):
    doc = recipe.copy()
    doc["embedding"] = get_embedding(recipe["description"])
    documents.append(doc)

collection.insert_many(documents)
print(f"\nInserted {len(documents)} recipes into MongoDB Atlas.")

Generating embeddings:   0%|          | 0/200 [00:00<?, ?it/s]


Inserted 200 recipes into MongoDB Atlas.


## 4. Create the Vector Index

In [10]:
# Create the vector index
search_index_model = SearchIndexModel(
    definition = {
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": EMBEDDING_DIMS,
                "similarity": "cosine"
            },
            {
                "type": "filter",
                "path": "cuisine"
            },
            {
                "type": "filter",
                "path": "difficulty"
            },
            {
                "type": "filter",
                "path": "prep_time"
            }
        ]
    },
    name = INDEX_NAME,
    type = "vectorSearch"
)

collection.create_search_index(model = search_index_model)
print(f"Index '{INDEX_NAME}' creation initiated.")

Index 'vector_index' creation initiated.


## 5. Wait for the Vector Index to be Ready

> **Note:** Atlas Vector Search reports the index status as `READY` before queries will actually return results. A fixed delay is not reliable - a small collection may be ready in five seconds, a larger one may need longer. The most robust approach is to poll the index status and then confirm with a test query.

After inserting data, confirm that the vector index we created is active.

In [11]:
print("Waiting for vector index to be active...")

while True:
    indexes = list(collection.list_search_indexes())
    status = next((idx["status"] for idx in indexes if idx["name"] == INDEX_NAME), None)
    if status == "READY":
        print(f"Index '{INDEX_NAME}' is active.")
        break
    print(f"  Status: {status} - waiting...")
    time.sleep(5)

# Confirm index is genuinely ready by running a test query
print("Confirming index is ready...")

while True:
    test = list(collection.aggregate([
        {
            "$vectorSearch": {
                "index":         INDEX_NAME,
                "path":          "embedding",
                "queryVector":   get_embedding("test"),
                "numCandidates": 10,
                "limit":         1,
            }
        }
    ]))
    if test:
        print("Index is ready.")
        break
    print("  Index not yet propagated - waiting...")
    time.sleep(5)

Waiting for vector index to be active...
  Status: PENDING - waiting...
  Status: PENDING - waiting...
  Status: PENDING - waiting...
  Status: PENDING - waiting...
  Status: PENDING - waiting...
Index 'vector_index' is active.
Confirming index is ready...
Index is ready.


## 6. Semantic Search

Atlas Vector Search uses an aggregation pipeline with a `$vectorSearch` stage. We embed the query and pass it to the pipeline, which returns the most similar recipes ranked by cosine similarity.

In [12]:
def search_recipes(query: str, top_k: int = 5):
    query_embedding = get_embedding(query)

    pipeline = [
        {
            "$vectorSearch": {
                "index":       INDEX_NAME,
                "path":        "embedding",
                "queryVector": query_embedding,
                "numCandidates": top_k * 10,
                "limit":       top_k,
            }
        },
        {
            "$project": {
                "_id":         0,
                "name":        1,
                "cuisine":     1,
                "difficulty":  1,
                "prep_time":   1,
                "description": 1,
                "score": {"$meta": "vectorSearchScore"},
            }
        }
    ]

    results = list(collection.aggregate(pipeline))
    print(f"\nQuery: '{query}'\n")
    for r in results:
        print(f"  {r['name']} ({r['cuisine']})")
        print(f"  {r['difficulty']} - {r['prep_time']} mins | Score: {r['score']:.3f}")
        print(f"  {r['description']}")
        print()

In [13]:
search_recipes("something warm and spicy for a cold evening")


Query: 'something warm and spicy for a cold evening'

  Braised Chicken With Eggplant (Ethiopian)
  Hard - 20 mins | Score: 0.785
  A a warm summer day favorite: warm and spicy braised chicken with eggplant and lentils.

  Braised Chicken Over Couscous (American)
  Easy - 20 mins | Score: 0.775
  Quick and warm and spicy: braised chicken with mushrooms over couscous, great for a dinner party.

  Chickpeas And Cauliflower Crusty Bread Bowl (Vietnamese)
  Hard - 30 mins | Score: 0.769
  Vietnamese classic: braised chickpeas with cauliflower, warm and spicy flavors ideal for a cozy night in.

  Chinese Halloumi And Eggplant (Chinese)
  Easy - 30 mins | Score: 0.766
  Quick and warm and spicy: roasted halloumi with eggplant over pasta, great for a weeknight dinner.

  Baked Chickpeas Over Flatbread (Indian)
  Medium - 90 mins | Score: 0.764
  A a cozy night in favorite: bold and aromatic baked chickpeas with roasted peppers and flatbread.



In [14]:
search_recipes("a light fresh dish with vegetables for summer")


Query: 'a light fresh dish with vegetables for summer'

  Poached Lentils Over Pasta (American)
  Hard - 60 mins | Score: 0.756
  A smoky and rich American dish featuring poached lentils with broccoli on a bed of pasta.

  Korean Chicken And Broccoli (Korean)
  Medium - 60 mins | Score: 0.756
  Simple and satisfying - grilled chicken tossed with broccoli and quinoa, light and fresh.

  Braised Chicken With Eggplant (Ethiopian)
  Hard - 20 mins | Score: 0.748
  A a warm summer day favorite: warm and spicy braised chicken with eggplant and lentils.

  Tuna And Carrots Rice Bowl (Mediterranean)
  Medium - 30 mins | Score: 0.747
  A smoky and rich Mediterranean dish featuring roasted tuna with carrots on a bed of rice.

  Steamed Beef Over Flatbread (Greek)
  Medium - 90 mins | Score: 0.746
  Quick and light and fresh: steamed beef with peas over flatbread, great for a warm summer day.



In [15]:
search_recipes("quick weeknight dinner with chicken and rice")


Query: 'quick weeknight dinner with chicken and rice'

  Korean Chicken And Broccoli (Korean)
  Medium - 60 mins | Score: 0.780
  Simple and satisfying - grilled chicken tossed with broccoli and quinoa, light and fresh.

  Ethiopian Style Lamb With Rice (Ethiopian)
  Hard - 45 mins | Score: 0.774
  A a light lunch favorite: bold and aromatic slow-cooked lamb with broccoli and rice.

  Simmered Chicken With Tomatoes (Middle Eastern)
  Easy - 30 mins | Score: 0.774
  Quick and tangy and bright: simmered chicken with tomatoes over quinoa, great for a weeknight dinner.

  Mexican Style Tuna With Quinoa (Mexican)
  Hard - 90 mins | Score: 0.764
  A a weeknight dinner favorite: warm and spicy braised tuna with roasted peppers and quinoa.

  Poached Chicken Over Pasta (Chinese)
  Medium - 20 mins | Score: 0.761
  Chinese classic: poached chicken with peas, hearty and satisfying flavors ideal for meal prep.



## 7. The Document Advantage - Filtered Search

Atlas Vector Search supports pre-filtering directly in the `$vectorSearch` stage using a `filter` parameter. Filters apply before the vector search, meaning only matching documents are considered as candidates. This is more precise than post-filtering and produces better results when filters are selective.

The filter fields must be declared in the index definition - we included `cuisine`, `difficulty` and `prep_time` when we created the index.

In [16]:
def search_recipes_filtered(
    query:      str,
    cuisine:    str  = None,
    difficulty: str  = None,
    max_time:   int  = None,
    top_k:      int  = 5
):
    query_embedding = get_embedding(query)

    # Build the filter document
    filter_doc = {}
    if cuisine:
        filter_doc["cuisine"] = {"$eq": cuisine}
    if difficulty:
        filter_doc["difficulty"] = {"$eq": difficulty}
    if max_time:
        filter_doc["prep_time"] = {"$lte": max_time}

    vector_search_stage = {
        "$vectorSearch": {
            "index":         INDEX_NAME,
            "path":          "embedding",
            "queryVector":   query_embedding,
            "numCandidates": top_k * 10,
            "limit":         top_k,
        }
    }
    if filter_doc:
        vector_search_stage["$vectorSearch"]["filter"] = filter_doc

    pipeline = [
        vector_search_stage,
        {
            "$project": {
                "_id":         0,
                "name":        1,
                "cuisine":     1,
                "difficulty":  1,
                "prep_time":   1,
                "description": 1,
                "score": {"$meta": "vectorSearchScore"},
            }
        }
    ]

    results = list(collection.aggregate(pipeline))
    label = f"query = '{query}'"
    if cuisine:    label += f", cuisine = '{cuisine}'"
    if difficulty: label += f", difficulty = '{difficulty}'"
    if max_time:   label += f", max_time = {max_time} mins"
    print(f"\n{label}\n")
    for r in results:
        print(f"  {r['name']} ({r['cuisine']})")
        print(f"  {r['difficulty']} - {r['prep_time']} mins | Score: {r['score']:.3f}")
        print(f"  {r['description']}")
        print()

In [17]:
# Easy Italian recipes ready in 30 minutes or less
search_recipes_filtered(
    "a comforting pasta dish",
    cuisine    = "Italian",
    difficulty = "Easy",
    max_time   = 30
)


query = 'a comforting pasta dish', cuisine = 'Italian', difficulty = 'Easy', max_time = 30 mins

  Steamed Lamb With Zucchini (Italian)
  Easy - 20 mins | Score: 0.749
  steamed lamb served over tortillas with zucchini, creamy and comforting and perfect for meal prep.

  Poached Chicken With Mushrooms (Italian)
  Easy - 20 mins | Score: 0.702
  Simple and satisfying - poached chicken tossed with mushrooms and flatbread, light and fresh.

  Chickpeas And Roasted Peppers Noodles Bowl (Italian)
  Easy - 15 mins | Score: 0.676
  Quick and zesty and vibrant: pan-fried chickpeas with roasted peppers over noodles, great for a dinner party.



In [18]:
# Quick Asian-inspired dishes
search_recipes_filtered(
    "bold spicy flavors with noodles",
    cuisine  = "Thai",
    max_time = 45
)


query = 'bold spicy flavors with noodles', cuisine = 'Thai', max_time = 45 mins

  Salmon And Roasted Peppers Flatbread Bowl (Thai)
  Easy - 20 mins | Score: 0.732
  Thai classic: steamed salmon with roasted peppers, sweet and savory flavors ideal for a weekend feast.

  Thai Lamb And Eggplant (Thai)
  Medium - 45 mins | Score: 0.730
  grilled lamb served over noodles with eggplant, hearty and satisfying and perfect for a weekend feast.

  Eggs And Kale Crusty Bread Bowl (Thai)
  Medium - 30 mins | Score: 0.714
  Quick and warm and spicy: steamed eggs with kale over crusty bread, great for a quick family meal.

  Thai Lamb And Tomatoes (Thai)
  Hard - 30 mins | Score: 0.696
  A entertaining guests favorite: light and fresh grilled lamb with tomatoes and pasta.

  Braised Lamb With Asparagus (Thai)
  Easy - 45 mins | Score: 0.683
  A light and fresh Thai dish featuring braised lamb with asparagus on a bed of crusty bread.



## 8. Exploring the Document Model

One of MongoDB's strengths is querying nested and array fields. We can search for recipes containing a specific ingredient using a standard MongoDB query - no joins, no separate table.

In [19]:
def find_by_ingredient(ingredient: str, limit: int = 5):
    results = collection.find(
        {"ingredients": {"$in": [ingredient]}},
        {"_id": 0, "name": 1, "cuisine": 1, "difficulty": 1, "prep_time": 1, "ingredients": 1}
    ).limit(limit)

    print(f"\nRecipes containing '{ingredient}':\n")
    for r in results:
        print(f"  {r['name']} ({r['cuisine']}) - {r['difficulty']}, {r['prep_time']} mins")
        print(f"  Ingredients: {', '.join(r['ingredients'])}")
        print()

In [20]:
find_by_ingredient("salmon")


Recipes containing 'salmon':

  Korean Chicken And Green Beans (Korean) - Hard, 90 mins
  Ingredients: peas, chicken, green beans, roasted peppers, couscous, salmon

  Salmon And Eggplant Tortillas Bowl (Italian) - Hard, 90 mins
  Ingredients: peas, asparagus, mushrooms, tortillas, carrots, salmon, spinach, sweet potato, eggplant, cauliflower

  Thai Style Salmon With Quinoa (Thai) - Medium, 30 mins
  Ingredients: lentils, lamb, roasted peppers, corn, carrots, salmon, quinoa, eggplant, halloumi

  Shrimp And Broccoli Couscous Bowl (Spanish) - Hard, 20 mins
  Ingredients: sweet potato, lamb, corn, couscous, shrimp, carrots, salmon, cauliflower, broccoli, tomatoes, chickpeas

  Eggs And Eggplant Noodles Bowl (Thai) - Medium, 60 mins
  Ingredients: zucchini, eggs, noodles, green beans, roasted peppers, kale, tofu, spinach, salmon, sweet potato, eggplant



In [21]:
find_by_ingredient("tofu")


Recipes containing 'tofu':

  Beef And Cauliflower Couscous Bowl (Ethiopian) - Easy, 90 mins
  Ingredients: tofu, couscous, beef, sweet potato, cauliflower

  Steamed Beef Over Flatbread (Greek) - Medium, 90 mins
  Ingredients: peas, flatbread, green beans, tofu, beef, broccoli, cauliflower

  Vietnamese Eggs And Asparagus (Vietnamese) - Medium, 45 mins
  Ingredients: eggs, noodles, asparagus, roasted peppers, tofu, sweet potato

  Thai Tofu And Carrots (Thai) - Medium, 60 mins
  Ingredients: lentils, eggs, roasted peppers, tofu, carrots, eggplant

  Japanese Style Halloumi With Lentils (Japanese) - Medium, 60 mins
  Ingredients: tofu, broccoli, lentils, halloumi



## Cleanup

In [22]:
client.close()
print("Connection closed.")

Connection closed.
